In [1]:
import scipy.linalg as la
import numpy as np
import copy
import time
import random
import matplotlib.pyplot as plt
import pandas as pd
import csv, ast
from qutip import *
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import KFold
from joblib import Parallel, delayed, dump, load
from extracter import *
from fitting_model import *
rng = np.random.default_rng(43)
np.random.seed(43)
random.seed(43)

In [70]:
fold_nums = [0,1,2,3,4]
num_fts = [36,49]
digits = [int(i) for i in range(10)]
rep = 1
all_targets = {}
all_states = {}
real_targets = {}
gamma='10' #.1

for fold_num in fold_nums:
    for num_ft in num_fts:
        ### creating the target 
        y_train = load(f'5_fold_{fold_num}_5speakers_{num_ft}fts.joblib')['y_train'] # unload y_train
        target_train = [int(i) for i in y_train][::num_ft] # store y_train while skipping the repetition
        y_test = load(f'5_fold_{fold_num}_5speakers_{num_ft}fts.joblib')['y_test'] # unload y_test
        target_test = [int(i) for i in y_test][::num_ft] # store y_test while skipping the repetition
        target_train.extend(target_test) # combine y_train,y_test as one
        name = str(fold_num) + '_' + str(num_ft) # each name indicates which fold and the amount of features
        real_targets[name] = target_train # store all targets into real_target dictionary

        ### Converting the digits into 10 classifiers and store them in all_targets
        for digit in digits:
            name = str(fold_num) + '_digit_' + str(digit) +'_'+ str(num_ft)
            all_targets[name] = []  
        w0 = 0
        w1 = 1
        for i in target_train:
            for j in digits:
                if int(j)==int(i):
                    name = str(fold_num) + '_digit_' + str(j) +'_'+ str(num_ft)
                    all_targets[name].append(w1)
                else:
                    name = str(fold_num) + '_digit_' + str(j) +'_'+ str(num_ft)
                    all_targets[name].append(w0)
        
     
        ### read Quantum features
        # read features from mean
        data = pd.read_csv(f"Performance/{num_ft}ASR_mean_fold{fold_num}.csv")#pd.read_csv(f"{num_ft}ASR_mean_{rep}rep_gamma{gamma}_fold{fold_num}.csv")
        name = str(fold_num)+'state_mean'+str(num_ft)
        skip_steps = rep
        X_mean= extract_block_single_column_skip(data ,column_name="P0",N_fts=num_ft,n_samples=500,skip_steps=skip_steps)
        all_states[name] = X_mean
         
        # read features from std
        data = pd.read_csv(f"Performance/{num_ft}ASR_std_fold{fold_num}.csv")
        name = str(fold_num)+'state_std'+str(num_ft)
        skip_steps = rep
        X_std= extract_block_single_column_skip(data ,column_name="P0",N_fts=num_ft,n_samples=500,skip_steps=skip_steps)
        all_states[name] = X_std


In [92]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
num_fts = [36,49]

train_points = 395
delay_points = 5
observation = train_points + delay_points
results = {}
errors = []
holder = []
for fold_num in fold_nums: 
    # formulate the final Features X
    for num_ft in num_fts: 
        ### Combine STD and Mean of each fTogether for computation
        name_state = str(fold_num)+'state_mean'+str(num_ft)
        states_mean = all_states[name_state]
        name_state = str(fold_num)+'state_std'+str(num_ft)
        states_std = all_states[name_state]
        states = np.concatenate((states_mean, states_std), axis=1) # combine std and mean
        holder.append(states) # hold 36 and 49 fts 
    states = np.concatenate((holder[1], holder[0]), axis=1) # combine fts 36 and 49
    holder = [] # reset holder 

    # extract train and test features
    X_train = states[delay_points:train_points+delay_points]
    X_test = states[train_points+delay_points:]
    for digit in digits: # for each digit
        name_target = str(fold_num) + '_digit_' + str(digit) +'_'+ str(num_ft) #NOTE: target of 36 is the same as 49. 
        target = all_targets[name_target] # get the target in 0,1 
        y_train = target[delay_points:train_points+delay_points] # separate into train and test
        y_test = target[train_points+delay_points:]
        W = fitting_function(X_train, y_train) # find the weight matrix
        y_prediction = predict(X_test, W) # find the final prediction
        results[name_target] = y_prediction 
        # results store all predictions for ten classifiers 

    # Convert results into actual labels
    prediction = []
    label = -1
    for i in range(len(y_prediction)):
        max_value = -10
        for value in results:
            label = value.split('_')[2] # classifier label
            fold = value.split('_')[0] # fold number
            fts = value.split('_')[3] # amount of features
            if results[value][i] > max_value and int(fold)==fold_num: 
                max_value = results[value][i]
                predicted_label = int(label)
        prediction.append(predicted_label)
    print("The predictions are: ", (prediction))
    real_answer = real_targets[str(fold_num)+'_'+str(num_ft)][observation:]
    print("The actual labels are: ", real_answer)

    # Convert to WER
    substitution = 0 
    for i in range(len(real_answer)):
        if real_answer[i] != prediction[i]:
            substitution += 1
    WER = substitution/len(real_answer)
    errors.append(WER)
    print("WER: " + str(WER))


    # Show the classification report
    real_answer = [str(i) for i in real_answer]
    prediction = [str(i) for i in prediction]
    print(f"Reports for fold {fold_num}:", classification_report(real_answer, prediction))

The predictions are:  [5, 5, 8, 4, 6, 8, 0, 6, 1, 2, 5, 2, 5, 6, 1, 3, 2, 8, 1, 6, 0, 8, 5, 5, 6, 6, 0, 4, 9, 5, 2, 6, 7, 7, 5, 3, 1, 2, 9, 7, 5, 3, 9, 4, 1, 4, 2, 6, 7, 8, 3, 0, 8, 8, 9, 9, 9, 4, 1, 9, 4, 8, 2, 6, 6, 9, 7, 9, 8, 8, 4, 5, 6, 3, 2, 6, 0, 3, 5, 4, 2, 1, 4, 2, 6, 9, 0, 4, 9, 7, 2, 1, 1, 8, 2, 4, 5, 7, 1, 4]
The actual labels are:  [9, 5, 7, 4, 6, 8, 0, 6, 1, 2, 5, 6, 5, 6, 1, 3, 2, 8, 1, 5, 0, 8, 5, 5, 6, 6, 0, 4, 9, 5, 2, 8, 7, 7, 5, 6, 1, 2, 9, 7, 5, 3, 9, 4, 1, 4, 2, 6, 7, 8, 3, 0, 8, 8, 9, 9, 9, 4, 1, 9, 7, 8, 2, 6, 6, 9, 9, 3, 8, 8, 4, 5, 6, 3, 2, 6, 0, 3, 5, 4, 2, 1, 4, 2, 6, 9, 0, 4, 9, 1, 2, 1, 1, 8, 6, 4, 5, 7, 1, 4]
WER: 0.11
Reports for fold 0:               precision    recall  f1-score   support

           0       1.00      1.00      1.00         6
           1       1.00      0.91      0.95        11
           2       0.83      1.00      0.91        10
           3       0.83      0.83      0.83         6
           4       0.92      1.00      0.96        

In [94]:
errors = np.asarray(errors, dtype=float)
mean_WER = np.mean(errors)
std_WER = np.std(errors, ddof=1)   # sample std
mean_accuracy = 1 - mean_WER
std_accuracy = std_WER
print("WERs:", errors)
print("Mean WER:", mean_WER)
print("Std WER:", std_WER)

print("Mean Accuracy:", mean_accuracy)
print("Std Accuracy:", std_accuracy)

WERs: [0.11 0.17 0.14 0.16 0.15]
Mean WER: 0.14600000000000002
Std WER: 0.02302172886644268
Mean Accuracy: 0.854
Std Accuracy: 0.02302172886644268
